In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

# Rutas del proyecto
DATA_DIR = Path("data")
FIG_DIR = DATA_DIR / "graficos"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def buscar_csv(nombre):
    """Busca primero en data/ y después en el directorio actual."""
    for ruta in (DATA_DIR / nombre, Path(nombre)):
        if ruta.exists():
            return ruta
    return DATA_DIR / nombre

# Cargar los datos
df_profes = pd.read_csv(buscar_csv("evaluaciones_profesorado.csv"))
df_llm = pd.read_csv(buscar_csv("resultados_benchmark.csv"))

print(f"Dimensiones Profesores: {df_profes.shape}")
print(f"Dimensiones LLMs: {df_llm.shape}")


In [ ]:
def aplanar_items(df):
    # Identificamos qué columnas son de información general (no notas de ítem)
    columnas_excluir = ['Nota_Global', 'Nota_Item_1', 'Nota_Item_2', 'Max_Score', 'Razonamiento_LLM']
    id_vars = [col for col in df.columns if col not in columnas_excluir]
    
    # Hacemos el "melt" (pasar columnas a filas)
    df_melt = df.melt(id_vars=id_vars, 
                      value_vars=['Nota_Item_1', 'Nota_Item_2'], 
                      var_name='Item_Num', 
                      value_name='Nota')
    
    # Eliminamos los valores nulos (tareas que no tenían Item_2)
    df_melt = df_melt.dropna(subset=['Nota']).copy()
    
    # Creamos un ID único para cada ítem (Ej: "cien_s1_t02_Item_1")
    df_melt['Item_ID'] = df_melt['Task_ID'] + "_" + df_melt['Item_Num'].str.replace('Nota_', '')
    
    return df_melt

# Aplicamos la función a ambos DataFrames
df_profes_items = aplanar_items(df_profes)
df_llm_items = aplanar_items(df_llm)

print(f"Total de ítems evaluados por profesores: {len(df_profes_items)}")
print(f"Total de ítems evaluados por LLMs: {len(df_llm_items)}")

In [ ]:
# Nos quedamos con las columnas clave de los profesores
columnas_agrupacion = ['Task_ID', 'Perfil_Alumno', 'Item_ID']

# Agrupamos y calculamos la moda (el valor más repetido)
# Usamos una función lambda. Si hay un caso rarísimo de empate total, coge el primer valor.
df_consenso = df_profes_items.groupby(columnas_agrupacion)['Nota'].apply(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
).reset_index()

# Renombramos la columna para que quede claro que es la nota de referencia
df_consenso.rename(columns={'Nota': 'Nota_Humana_Real'}, inplace=True)

# Opcional: Ver si algún humano puso una nota que no sea 0 o 1 (por si hubo errores al teclear en el CSV)
print("Valores únicos en el consenso humano:", df_consenso['Nota_Humana_Real'].unique())
df_consenso.head()

In [ ]:
# Asegurarnos de que las notas de la IA sean numéricas
df_llm_items['Nota'] = pd.to_numeric(df_llm_items['Nota'], errors='coerce')

# Detectar alucinaciones
df_llm_items['Alucinacion_Formato'] = ~df_llm_items['Nota'].isin([0.0, 1.0])

# Clasificar el tipo de error
def tipo_error(nota):
    if pd.isna(nota): return "Error_No_Numerico"
    elif nota in [0.0, 1.0]: return "Correcto"
    elif nota > 1.0 or nota < 0.0: return "Fuera_De_Rango"
    else: return "Decimal_No_Permitido"

df_llm_items['Tipo_Error_Formato'] = df_llm_items['Nota'].apply(tipo_error)

# FILTRAR: Nos quedamos solo con las filas que NO son "Correcto"
df_alucinaciones = df_llm_items[df_llm_items['Tipo_Error_Formato'] != 'Correcto']

print(f"Se han detectado {len(df_alucinaciones)} alucinaciones de formato en total.\n")
print("Detalle de las alucinaciones:")

# Mostramos Modelo, Tarea, Ítem, Perfil, la nota loca que puso y el tipo de error
columnas_ver = ['Modelo_Evaluador', 'Task_ID', 'Item_ID', 'Perfil_Alumno', 'Nota', 'Tipo_Error_Formato']
display(df_alucinaciones[columnas_ver].sort_values(by=['Tipo_Error_Formato', 'Modelo_Evaluador']))

In [ ]:
# Unimos el dataframe de las IAs con el dataframe del consenso humano
df_master = pd.merge(df_llm_items, df_consenso, 
                     on=['Task_ID', 'Perfil_Alumno', 'Item_ID'], 
                     how='left')

# Calculamos si la IA acertó. 
# Condiciones para acertar: Que la nota sea exactamente igual a la humana Y que NO sea una alucinación
df_master['Acierto'] = (df_master['Nota'] == df_master['Nota_Humana_Real']) & (~df_master['Alucinacion_Formato'])

# Clasificar si es un Falso Positivo o Falso Negativo (súper útil para luego)
def clasificar_fallo(row):
    if row['Acierto']:
        return "Acierto"
    elif row['Alucinacion_Formato']:
        return "Fallo_Por_Alucinacion"
    elif row['Nota'] == 1.0 and row['Nota_Humana_Real'] == 0.0:
        return "Falso_Positivo (Demasiado Generoso)"
    elif row['Nota'] == 0.0 and row['Nota_Humana_Real'] == 1.0:
        return "Falso_Negativo (Demasiado Estricto)"
    else:
        return "Otro_Error"

df_master['Tipo_Resultado'] = df_master.apply(clasificar_fallo, axis=1)

# Mostramos cómo queda nuestro dataset maestro listo para analizar
df_master.head()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.stats.inter_rater as irr

# 1. Preparar los datos (Pivotar para tener a los 3 profesores en columnas)
df_pivot_profes = df_profes_items.pivot_table(
    index=['Task_ID', 'Item_ID', 'Perfil_Alumno'], 
    columns='Modelo_Evaluador', 
    values='Nota', 
    aggfunc='first'
).reset_index()

# =====================================================================
# MÉTRICA 1: UNANIMIDAD GLOBAL
# =====================================================================
df_pivot_profes['Unanimidad'] = (df_pivot_profes['ev1'] == df_pivot_profes['ev2']) & (df_pivot_profes['ev2'] == df_pivot_profes['ev3'])

total_respuestas = len(df_pivot_profes)
unanimidad_total = df_pivot_profes['Unanimidad'].sum()
porcentaje_unanimidad = (unanimidad_total / total_respuestas) * 100

print("======================================================")
print(" 1. MÉTRICA DE UNANIMIDAD GLOBAL")
print("======================================================")
print(f"Total de respuestas evaluadas: {total_respuestas}")
print(f"Casos con unanimidad total (los 3 de acuerdo): {unanimidad_total}")
print(f"Porcentaje de unanimidad: {porcentaje_unanimidad:.2f}%\n")


# =====================================================================
# MÉTRICA 2: KAPPA DE FLEISS AGRUPADO POR PERFIL DE ALUMNO
# =====================================================================
kappas_perfil = []

for perfil in df_pivot_profes['Perfil_Alumno'].unique():
    subset = df_pivot_profes[df_pivot_profes['Perfil_Alumno'] == perfil][['ev1', 'ev2', 'ev3']].dropna()
    
    if len(subset) > 0:
        # CORRECCIÓN: Usamos .iloc[0] en lugar de [0] para evitar el FutureWarning de Pandas
        if subset.nunique().nunique() == 1 and subset.nunique().iloc[0] == 1:
            kappas_perfil.append({'Perfil_Alumno': perfil, 'Kappa_Fleiss': 1.0})
        else:
            agg = irr.aggregate_raters(subset)[0]
            try:
                import warnings
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    kappa_val = irr.fleiss_kappa(agg, method='fleiss')
                
                if np.isnan(kappa_val):
                    kappa_val = 1.0
                    
                kappas_perfil.append({'Perfil_Alumno': perfil, 'Kappa_Fleiss': kappa_val})
            except:
                pass

df_kappas_perfil = pd.DataFrame(kappas_perfil).sort_values(by='Kappa_Fleiss', ascending=True)

print("======================================================")
print(" 2. ACUERDO HUMANO POR PERFIL (Kappa de Fleiss)")
print("======================================================")
display(df_kappas_perfil.style.format({'Kappa_Fleiss': '{:.3f}'}).hide(axis="index"))
print("\n")


# =====================================================================
# MÉTRICA 3: KAPPA DE FLEISS AGRUPADO POR ÍTEM
# =====================================================================
kappas_item = []

for item in df_pivot_profes['Item_ID'].unique():
    subset = df_pivot_profes[df_pivot_profes['Item_ID'] == item][['ev1', 'ev2', 'ev3']].dropna()
    
    if len(subset) > 0:
        # CORRECCIÓN: Usamos .iloc[0] en lugar de [0] para evitar el FutureWarning de Pandas
        if subset.nunique().nunique() == 1 and subset.nunique().iloc[0] == 1:
            kappas_item.append({'Item_ID': item, 'Kappa_Fleiss': 1.0})
        else:
            agg = irr.aggregate_raters(subset)[0]
            try:
                import warnings
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    kappa_val = irr.fleiss_kappa(agg, method='fleiss')
                
                if np.isnan(kappa_val):
                    kappa_val = 1.0
                    
                kappas_item.append({'Item_ID': item, 'Kappa_Fleiss': kappa_val})
            except:
                pass

df_kappas_item = pd.DataFrame(kappas_item).sort_values(by='Kappa_Fleiss', ascending=True)

print("======================================================")
print(" 3. ACUERDO HUMANO POR ÍTEM (Kappa de Fleiss)")
print("======================================================")
display(df_kappas_item.style.format({'Kappa_Fleiss': '{:.3f}'}).hide(axis="index"))

In [ ]:
import pandas as pd

# =====================================================================
# 1. CREAR EL "GROUND TRUTH" (Consenso Humano por Voto Mayoritario)
# =====================================================================
df_pivot_profes['Nota_Humana_Real'] = df_pivot_profes[['ev1', 'ev2', 'ev3']].mode(axis=1)[0]
df_consenso = df_pivot_profes[['Task_ID', 'Item_ID', 'Perfil_Alumno', 'Nota_Humana_Real']]

# =====================================================================
# 2. CRUZAR DATOS Y CALCULAR EL ACIERTO (ACCURACY) DE LAS IAs
# =====================================================================
df_master = pd.merge(
    df_llm_items,
    df_consenso,
    on=['Task_ID', 'Item_ID', 'Perfil_Alumno'],
    how='inner'
)

df_master['Nota'] = pd.to_numeric(df_master['Nota'], errors='coerce')
df_master['Acierto'] = (
    (df_master['Nota'] == df_master['Nota_Humana_Real'])
    & (df_master['Nota'].isin([0.0, 1.0]))
)

df_ranking = df_master.groupby('Modelo_Evaluador')['Acierto'].mean() * 100
df_ranking = df_ranking.reset_index().sort_values(by='Acierto', ascending=False)

print("======================================================")
print(" RANKING DE RENDIMIENTO DE LOS MODELOS (ACCURACY)")
print("======================================================")
display(df_ranking.style.format({'Acierto': '{:.2f}%'}).hide(axis="index"))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# =====================================================================
# CONFIGURACIÓN DE ESTILO ACADÉMICO (TFG)
# =====================================================================
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.autolayout': True
})

# =====================================================================
# PARTE 1: ANÁLISIS DE EFICIENCIA (Tiempo vs. Precisión)
# =====================================================================
df_tiempos = df_master.groupby('Modelo_Evaluador')['Tiempo_Respuesta_s'].mean().reset_index()
df_eficiencia = pd.merge(df_ranking, df_tiempos, on='Modelo_Evaluador')

print("======================================================")
print(" 1. EFICIENCIA: TIEMPO MEDIO vs. ACIERTO")
print("======================================================")
display(df_eficiencia.style.format({'Acierto': '{:.2f}%', 'Tiempo_Respuesta_s': '{:.2f} s'}).hide(axis="index"))
print("\n")

# Gráfico de Dispersión (Scatter Plot) de Eficiencia
fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(
    data=df_eficiencia, 
    x='Tiempo_Respuesta_s', 
    y='Acierto', 
    hue='Modelo_Evaluador', 
    s=250, 
    palette='tab10',
    edgecolor='black',
    legend=False,
    ax=ax
)

# Añadimos las etiquetas de los modelos al lado de cada punto
for i, row in df_eficiencia.iterrows():
    ax.text(row['Tiempo_Respuesta_s'] + 1, row['Acierto'] - 0.5, 
            row['Modelo_Evaluador'], fontsize=11, fontweight='bold', color='#333333')



ax.set_xlabel('Tiempo Medio de Respuesta (segundos)')
ax.set_ylabel('Tasa de Acierto Global (%)')

sns.despine()
plt.savefig(FIG_DIR / 'grafica_5_eficiencia_tfg.png', dpi=300, bbox_inches='tight')
plt.show()




In [ ]:
import statsmodels.stats.inter_rater as irr

# 1. Cogemos todas las evaluaciones de los 3 profesores de golpe y quitamos los nulos
subset_global = df_pivot_profes[['ev1', 'ev2', 'ev3']].dropna()

# 2. Formateamos los datos para la fórmula matemática
agg_global = irr.aggregate_raters(subset_global)[0]

# 3. Calculamos el Kappa de Fleiss global
kappa_global = irr.fleiss_kappa(agg_global, method='fleiss')

# 4. Función para interpretar el resultado según Landis y Koch (1977)
def interpretar_kappa(k):
    if k < 0: return "Desacuerdo"
    elif k <= 0.20: return "Acuerdo Leve"
    elif k <= 0.40: return "Acuerdo Justo"
    elif k <= 0.60: return "Acuerdo Moderado"
    elif k <= 0.80: return "Acuerdo Sustancial"
    else: return "Acuerdo Casi Perfecto"

interpretacion = interpretar_kappa(kappa_global)

print("======================================================")
print(" 0. ACUERDO HUMANO GLOBAL (KAPPA DE FLEISS)")
print("======================================================")
print(f"Total de evaluaciones analizadas: {len(subset_global)}")
print(f"Índice Kappa de Fleiss Global: {kappa_global:.3f}")
print(f"Interpretación científica: {interpretacion}")
print("======================================================")

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# 1. EVALUADORES HUMANOS vs. CONSENSO EXCLUYENTE (Los otros 2)
# =====================================================================
human_metrics = []

# Diccionario para saber quiénes son los "otros dos" para cada evaluador
otros_evaluadores = {
    'ev1': ['ev2', 'ev3'],
    'ev2': ['ev1', 'ev3'],
    'ev3': ['ev1', 'ev2']
}

for ev, otros in otros_evaluadores.items():
    # 1. Filtramos SOLO los casos donde los otros dos profesores están de acuerdo
    consenso_excluyente = df_pivot_profes[df_pivot_profes[otros[0]] == df_pivot_profes[otros[1]]].copy()
    consenso_excluyente = consenso_excluyente.dropna(subset=[ev, otros[0]])
    
    # 2. La "Verdad Absoluta" es la nota en la que los otros dos coincidieron
    verdad_otros = consenso_excluyente[otros[0]]
    # 3. La predicción es lo que votó el evaluador que estamos analizando
    prediccion_ev = consenso_excluyente[ev]
    
    # 4. Calculamos métricas sobre ese subconjunto
    acc = accuracy_score(verdad_otros, prediccion_ev)
    kappa = cohen_kappa_score(verdad_otros, prediccion_ev)
    
    human_metrics.append({
        'Evaluador / Modelo': f'Humano: {ev.upper()}',
        'Accuracy (%)': acc * 100,
        'Kappa de Cohen': kappa,
        'Tipo': 'Humano'
    })

df_human_metrics = pd.DataFrame(human_metrics)

print("======================================================")
print(" 1. HUMANOS VS. CONSENSO EXCLUYENTE (Leave-One-Out)")
print("======================================================")
display(df_human_metrics[['Evaluador / Modelo', 'Accuracy (%)', 'Kappa de Cohen']].style.format({
    'Accuracy (%)': '{:.2f}%',
    'Kappa de Cohen': '{:.3f}'
}).hide(axis="index"))
print("\n")


# =====================================================================
# 2. MODELOS LLM vs. CONSENSO GLOBAL MAYORITARIO
# =====================================================================
llm_metrics = []
df_master_kappa = df_master.copy()

# Castigamos alucinaciones poniéndolas a -1 (para que cuenten como fallo y penalicen el Kappa)
df_master_kappa['Nota_Procesada'] = df_master_kappa['Nota'].apply(lambda x: x if x in [0.0, 1.0] else -1)

for modelo in df_master_kappa['Modelo_Evaluador'].unique():
    subset_llm = df_master_kappa[df_master_kappa['Modelo_Evaluador'] == modelo]
    
    acc = accuracy_score(subset_llm['Nota_Humana_Real'], subset_llm['Nota_Procesada'])
    kappa = cohen_kappa_score(subset_llm['Nota_Humana_Real'], subset_llm['Nota_Procesada'])
    
    llm_metrics.append({
        'Evaluador / Modelo': modelo,
        'Accuracy (%)': acc * 100,
        'Kappa de Cohen': kappa,
        'Tipo': 'IA'
    })

df_llm_metrics = pd.DataFrame(llm_metrics).sort_values(by='Kappa de Cohen', ascending=False)

print("======================================================")
print(" 2. MODELOS LLM VS. CONSENSO GLOBAL")
print("======================================================")
display(df_llm_metrics[['Evaluador / Modelo', 'Accuracy (%)', 'Kappa de Cohen']].style.format({
    'Accuracy (%)': '{:.2f}%',
    'Kappa de Cohen': '{:.3f}'
}).hide(axis="index"))
print("\n")


# =====================================================================
# 3. GRÁFICO COMBINADO: HUMANOS Y LLMs (Comparativa de Kappa)
# =====================================================================
df_combinado = pd.concat([df_human_metrics, df_llm_metrics]).sort_values(by='Kappa de Cohen', ascending=False)

plt.rcParams.update({'font.family': 'serif', 'font.size': 12, 'figure.autolayout': True})
fig, ax = plt.subplots(figsize=(10, 6))

colores = ['#4daf4a' if tipo == 'Humano' else '#1f77b4' for tipo in df_combinado['Tipo']]

sns.barplot(
    data=df_combinado, 
    x='Kappa de Cohen', 
    y='Evaluador / Modelo', 
    hue='Evaluador / Modelo',
    palette=colores,
    edgecolor='black',
    legend=False,
    ax=ax
)

ax.axvline(x=0.8, color='black', linestyle='--', alpha=0.5, label='Acuerdo Casi Perfecto (0.8)')
ax.axvline(x=0.6, color='black', linestyle=':', alpha=0.5, label='Acuerdo Sustancial (0.6)')

ax.set_xlabel('Índice Kappa de Cohen (Concordancia con el Consenso)')
ax.set_ylabel('Evaluador / Modelo')
ax.set_xlim(0, 1.05)

# Crear leyenda manual para explicar los colores
from matplotlib.patches import Patch
leyenda_elementos = [
    Patch(facecolor='#4daf4a', edgecolor='black', label='Profesores Humanos (vs Excluyente)'),
    Patch(facecolor='#1f77b4', edgecolor='black', label='Modelos IA (vs Global)')
]
ax.legend(handles=leyenda_elementos, loc='lower right')

sns.despine()
plt.savefig(FIG_DIR / 'grafica_7_kappa_humanos_vs_ia_tfg.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# 1. CLASIFICAR LOS TIPOS DE ERRORES DE LA IA
# =====================================================================
# Creamos una copia para no tocar el original
df_sesgos = df_master.copy()

# Nos aseguramos de que la nota sea numérica
df_sesgos['Nota'] = pd.to_numeric(df_sesgos['Nota'], errors='coerce')

def clasificar_error(row):
    if row['Acierto']:
        return 'Acierto'
    elif row['Nota'] not in [0.0, 1.0]:
        return 'Alucinación'
    elif row['Nota'] == 1.0 and row['Nota_Humana_Real'] == 0.0:
        return 'Engañado (Falso Positivo)'
    elif row['Nota'] == 0.0 and row['Nota_Humana_Real'] == 1.0:
        return 'Demasiado Estricto (Falso Negativo)'
    else:
        return 'Otro Error'

df_sesgos['Tipo_Error'] = df_sesgos.apply(clasificar_error, axis=1)

# =====================================================================
# 2. CALCULAR TASA DE FALLO POR PERFIL DE ALUMNO
# =====================================================================
# Calculamos el % de veces que la IA NO acertó, agrupado por Modelo y Perfil
df_fallos = df_sesgos.assign(Fallo=~df_sesgos['Acierto']).groupby(['Modelo_Evaluador', 'Perfil_Alumno'])['Fallo'].mean().reset_index()
df_fallos['Fallo'] = df_fallos['Fallo'] * 100 # Pasar a porcentaje

# Pivotar para el mapa de calor
df_heatmap_fallos = df_fallos.pivot(index='Modelo_Evaluador', columns='Perfil_Alumno', values='Fallo')

# =====================================================================
# 3. GRÁFICO: MAPA DE CALOR DE VULNERABILIDADES (TFG)
# =====================================================================
plt.rcParams.update({'font.family': 'serif', 'font.size': 12, 'figure.autolayout': True})
fig, ax = plt.subplots(figsize=(10, 6))

# Mapa de calor (Rojo = Más fallos, Blanco = Menos fallos)
sns.heatmap(
    df_heatmap_fallos, 
    annot=True, 
    fmt=".1f", 
    cmap="Reds", 
    cbar_kws={'label': 'Tasa de Fallo (%)'},
    linewidths=1,
    linecolor='black',
    ax=ax
)

ax.set_xlabel('Perfil del Alumno')
ax.set_ylabel('Modelo de Lenguaje (LLM)')


plt.savefig(FIG_DIR / 'grafica_8_sesgos_perfil_tfg.png', dpi=300, bbox_inches='tight')
plt.show()

# =====================================================================
# 4. TABLA DETALLADA DEL PERFIL "TRAMPA" Y "BORDE" (Opcional para comentar)
# =====================================================================
print("======================================================")
print(" DESGLOSE DE ERRORES EN PERFILES PROBLEMÁTICOS")
print("======================================================")
# Filtramos solo los errores (sin contar aciertos) para ver de qué tipo son
df_solo_errores = df_sesgos[df_sesgos['Tipo_Error'] != 'Acierto']
desglose = pd.crosstab(df_solo_errores['Perfil_Alumno'], df_solo_errores['Tipo_Error'])
display(desglose)

In [ ]:
import pandas as pd

# =====================================================================
# TABLA DE DISTRIBUCIÓN DE NOTAS (0 y 1) POR PERFIL DE ALUMNO
# =====================================================================
# Usamos pd.crosstab para cruzar los perfiles con la nota del consenso humano
distribucion_notas = pd.crosstab(df_consenso['Perfil_Alumno'], df_consenso['Nota_Humana_Real'])

# Renombramos las columnas para que quede más profesional
distribucion_notas.columns = ['Cantidad de 0s (Suspensos)', 'Cantidad de 1s (Aprobados)']

# Añadimos una columna de totales para ver cuántos ítems tiene cada perfil
distribucion_notas['Total Ítems Evaluados'] = distribucion_notas['Cantidad de 0s (Suspensos)'] + distribucion_notas['Cantidad de 1s (Aprobados)']

# Opcional: Añadir porcentajes
distribucion_notas['% de 1s'] = (distribucion_notas['Cantidad de 1s (Aprobados)'] / distribucion_notas['Total Ítems Evaluados']) * 100

print("==================================================================")
print(" DISTRIBUCIÓN DEL GROUND TRUTH: 0s y 1s POR PERFIL DE ALUMNO")
print("==================================================================")
# Mostramos la tabla con un pequeño gradiente de color y los porcentajes formateados
display(distribucion_notas.style.format({
    '% de 1s': '{:.1f}%'
}).background_gradient(cmap='Blues', subset=['Cantidad de 0s (Suspensos)', 'Cantidad de 1s (Aprobados)']))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import math

# =====================================================================
# MATRICES DE CONFUSIÓN POR MODELO (Cuadrícula TFG)
# =====================================================================
# Obtenemos la lista de modelos
modelos = df_master_kappa['Modelo_Evaluador'].unique()
n_modelos = len(modelos)

# Configuramos la cuadrícula (ej. 4 columnas, y las filas que hagan falta)
cols = 4
rows = math.ceil(n_modelos / cols)

# Ajustar estilo para que quepa bien en el TFG
plt.rcParams.update({'font.family': 'serif', 'font.size': 11, 'figure.autolayout': False})
fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))

# Aplanamos el array de ejes para iterar fácilmente, por si hay varias filas
if n_modelos > 1:
    axes = axes.flatten()
else:
    axes = [axes]

for i, modelo in enumerate(modelos):
    # Filtramos los datos solo para este modelo
    subset = df_master_kappa[df_master_kappa['Modelo_Evaluador'] == modelo]
    
    # Calculamos la matriz de confusión (Aseguramos el orden 0.0 primero, 1.0 después)
    cm = confusion_matrix(subset['Nota_Humana_Real'], subset['Nota_Procesada'], labels=[0.0, 1.0])
    
    # Dibujamos el Heatmap en su casilla correspondiente
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', cbar=False, ax=axes[i],
                xticklabels=['Suspenso (0)', 'Aprobado (1)'],
                yticklabels=['Suspenso (0)', 'Aprobado (1)'],
                linewidths=1, linecolor='black')
    
    # Títulos y etiquetas de cada subgráfico
    axes[i].set_title(f'{modelo}', fontweight='bold', pad=10)
    axes[i].set_xlabel('Predicción de la IA')
    axes[i].set_ylabel('Realidad (Consenso Humano)')

# Si sobran casillas en la cuadrícula (ej. hay 7 modelos y 8 huecos), las ocultamos
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

# Ajustar el espaciado para que no se pisen los textos
plt.tight_layout()

# Guardar en alta calidad
plt.savefig(FIG_DIR / 'grafica_10_matrices_confusion_tfg.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# =====================================================================
# MATRICES DE CONFUSIÓN: TOP 3 MODELOS
# =====================================================================
# Definimos exactamente los 3 modelos que queremos analizar
modelos_top = ['llama3.1:8b', 'qwen3-vl:30b', 'poligpt-code']

# Ajustamos el estilo para el TFG
plt.rcParams.update({'font.family': 'serif', 'font.size': 11, 'figure.autolayout': False})

# Configuramos la cuadrícula: 1 fila, 3 columnas
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, modelo in enumerate(modelos_top):
    # Filtramos los datos solo para este modelo
    subset = df_master_kappa[df_master_kappa['Modelo_Evaluador'] == modelo]
    
    # Calculamos la matriz de confusión (Aseguramos el orden 0.0 primero, 1.0 después)
    cm = confusion_matrix(subset['Nota_Humana_Real'], subset['Nota_Procesada'], labels=[0.0, 1.0])
    
    # Dibujamos el Heatmap en su casilla correspondiente
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', cbar=False, ax=axes[i],
                xticklabels=['Suspenso (0)', 'Aprobado (1)'],
                yticklabels=['Suspenso (0)', 'Aprobado (1)'],
                linewidths=1, linecolor='black')
    
    # Títulos y etiquetas de cada subgráfico
    axes[i].set_title(f'Modelo: {modelo}', fontweight='bold', pad=10)
    axes[i].set_xlabel('Predicción de la IA')
    
    # Solo ponemos la etiqueta Y en el primer gráfico para no recargar visualmente
    if i == 0:
        axes[i].set_ylabel('Realidad (Consenso Humano)')
    else:
        axes[i].set_ylabel('')

# Ajustar el espaciado para que quede impecable
plt.tight_layout()

# Guardar la gráfica en alta calidad
plt.savefig(FIG_DIR / 'grafica_10_matrices_confusion_top3_tfg.png', dpi=300, bbox_inches='tight')
plt.show()